### text tokenizing

#### “The Verdict,” a short story by Edith Wharton

In [3]:
# load the file

import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x10f9d4770>)

In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f'total number of chars: {len(raw_text)}')
print(raw_text[:100])

total number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [5]:
import re

#### from w3: The strip() method removes any leading, and trailing whitespaces.

#### Leading means at the beginning of the string, trailing means at the end.


In [7]:
preprocessed = re.split(r'([,.()_]"|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed)

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that,', 'in', 'the', 'height', 'of', 'his', 'glory,', 'he', 'had', 'dropped', 'his', 'painting,', 'married', 'a', 'rich', 'widow,', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera.', '(Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence.)', '"The', 'height', 'of', 'his', 'glory"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it.', 'I', 'can', 'hear', 'Mrs.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--', 'deploring', 'his', 'unaccountable', 'abdication.', '"Of', 'course', "it's", 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'way", 'up;', 'but', 'I', "don't", 'think', 'of', 'that,', 'Mr.', 'Rickham', '--', 'the', 'loss', 'to', 'Arrt', 'is', 'all', 'I', 'think', '

In [8]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f'vocab size: {vocab_size}')

vocab size: 1412


In [9]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [10]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.()_]"|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        enc_ids = [self.str_to_int[s] for s in preprocessed]
        return enc_ids
    def decode (self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) 
        return text

In [11]:
tokenizer = SimpleTokenizerV1(vocab)
text = "thought Jack Gisburn rather a cheap genius"
ids = tokenizer.encode(text)
print(f'ids: {ids}')

ids: [1246, 118, 94, 1034, 204, 368, 618]


In [12]:
decoded_text = tokenizer.decode(ids)
print(decoded_text)

thought Jack Gisburn rather a cheap genius


#### if any keyword that is not present in the story is used then that return an error. To resolve it, we add special context tokens

In [14]:
all_tokens = sorted(list(preprocessed))
all_tokens.extend(['<|endoftext|>','<|unk|>'])
vocab = {token:integ for integ, token in enumerate(all_tokens)}
print(len(vocab.items()))

1414


In [15]:
for i, v in enumerate(list(vocab.items())[-5:]):
    print(v)

('younger', 3858)
('your', 3861)
('yourself', 3862)
('<|endoftext|>', 3863)
('<|unk|>', 3864)


In [16]:
vocab

{'"': 0,
 '"Ah': 1,
 '"Ah,': 2,
 '"Be': 4,
 '"Begin': 5,
 '"By': 6,
 '"Come': 7,
 '"Destroyed': 8,
 '"Don\'t': 9,
 '"Gisburns"': 10,
 '"Grindles': 11,
 '"Hang': 12,
 '"Has': 13,
 '"How': 14,
 '"I': 21,
 '"I\'d': 22,
 '"If': 24,
 '"It': 25,
 '"It\'s': 27,
 '"Jack': 28,
 '"Money\'s': 29,
 '"Moon-dancers"': 30,
 '"Mr.': 31,
 '"Mrs.': 32,
 '"My': 33,
 '"Never': 35,
 '"Of': 36,
 '"Oh,': 41,
 '"Once,': 42,
 '"Only': 43,
 '"Or': 44,
 '"That': 45,
 '"The': 47,
 '"Then': 48,
 '"There': 49,
 '"There:': 50,
 '"This': 51,
 '"We': 52,
 '"Well,': 54,
 '"What': 55,
 '"When': 58,
 '"Why': 59,
 '"Yes': 61,
 '"Yes,': 62,
 '"You': 63,
 '"but': 64,
 '"deadening': 65,
 '"dragged': 66,
 '"effects";': 67,
 '"interesting":': 68,
 '"lift': 69,
 '"obituary"': 70,
 '"strongest': 71,
 '"strongly"': 72,
 '"sweetly"': 73,
 "'Are": 74,
 "'It's": 75,
 "'coming'": 76,
 "'done'": 77,
 "'subject.'": 78,
 "'technique'": 79,
 "'way": 80,
 '(I': 82,
 '(Though': 83,
 ',"': 95,
 '--': 192,
 '.': 212,
 '."': 236,
 'A': 238,
 

In [17]:
class SimpleTokenTokenizerV2:
    def __init__(self, vocab):
        self.int_to_str = {i:s for i,s in vocab.items()}
        self.str_to_int = vocab
    def encode(self, text):
        preprocessed = re.split(r'([,.()_]"|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else '<|unk|>' for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) 
        return text
        

In [18]:
tokenizer = SimpleTokenTokenizerV2(vocab)
text = "thought Jack Gisburn rather a cheap genius, hello world"
ids = tokenizer.encode(text)
print(f'ids: {ids}')

ids: [3369, 461, 292, 2786, 705, 1129, 3864, 3864, 3864]


### byte pair encoding

In [20]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [21]:
tokenizer = tiktoken.get_encoding("gpt2")

In [22]:
text = "it is very cold today. <|endoftext|> let's go to a beach"
tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[270, 318, 845, 4692, 1909, 13, 220, 50256, 1309, 338, 467, 284, 257, 10481]

In [23]:
tokenizer.decode(tokenizer.encode(text, allowed_special={"<|endoftext|>"}))

"it is very cold today. <|endoftext|> let's go to a beach"

In [24]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [25]:
enc_sample = enc_text[50:]

In [26]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(x,y)

[290, 4920, 2241, 287] [4920, 2241, 287, 257]


In [27]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f'context- > {context}, desired - > {desired}')

context- > [290], desired - > 4920
context- > [290, 4920], desired - > 2241
context- > [290, 4920, 2241], desired - > 287
context- > [290, 4920, 2241, 287], desired - > 257


In [28]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f'{tokenizer.decode(context)} - > {tokenizer.decode([desired])}')

 and - >  established
 and established - >  himself
 and established himself - >  in
 and established himself in - >  a


In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

In [30]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
            

In [31]:
def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [32]:
with open('the-verdict.txt','r') as f:
    raw_text = f.read()

dataloader = create_dataloader(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)


[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


In [33]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [34]:
max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("token ids: \n", inputs)
print("inputs shape: \n", inputs.shape)

token ids: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
inputs shape: 
 torch.Size([8, 4])


In [35]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [36]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embedding = pos_embedding_layer(torch.arange(context_length))
print(pos_embedding.shape)

torch.Size([4, 256])


In [37]:
input_embeddings = token_embeddings + pos_embedding
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [81]:
input_embeddings[0]

tensor([[-1.4459, -0.0620, -0.3277,  ..., -0.2841,  0.9057, -1.0574],
        [ 1.1011,  2.0707,  0.8525,  ...,  1.2509,  1.6314,  1.7848],
        [ 0.0325,  1.4512,  0.7431,  ...,  2.5921,  0.6675, -1.2268],
        [-1.8343, -0.1094,  0.6378,  ..., -0.0354, -1.0483, -0.8347]],
       grad_fn=<SelectBackward0>)

In [85]:
import numpy as np

In [87]:
np.dot([0.43, 0.15, 0.89],[0.55, 0.87, 0.66])

0.9544

In [89]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [91]:
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])

In [93]:
attn_scores_2

tensor([0., 0., 0., 0., 0., 0.])

In [95]:
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [97]:
attn_scores_2_tmp = attn_scores_2/ attn_scores_2.sum()
print(attn_scores_2_tmp)

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [99]:
def softmax_naive(x):
    return torch.exp(x)/ torch.exp(x).sum(dim = 0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [101]:
# or
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

In [107]:
print('attn weights: ',attn_weights_2)

attn weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [111]:
torch.zeros(query.shape)

tensor([0., 0., 0.])

In [117]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2+= x_i*attn_weights_2[i]
print('context vector: ',context_vec_2)

context vector:  tensor([0.4419, 0.6515, 0.5683])


In [119]:
attn_scores = torch.empty(6,6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i][j] = torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [121]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [123]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [125]:
all_context_vecs = attn_weights@inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
